# Notebook 01 — Inspección Inicial
## Proyecto Integrador | Minería de Datos I
**Integrantes:** Valeria Martinetti y Santiago Gallardo

**Dataset:** `streaming_users_dirty.json` — Usuarios de plataforma de streaming

---
### Objetivo
Reunir evidencia sobre la estructura y el estado del dataset sin tomar decisiones de limpieza todavía. El objetivo es responder: ¿qué tenemos?, ¿qué problemas se observan?, ¿qué preguntas se abren?

In [105]:
import pandas as pd

df = pd.read_json('../data/raw/streaming_users_dirty.json')
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

Filas: 8160 | Columnas: 8


## 1. Estructura general

In [106]:
# Primeras filas
df.head(10)

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1
5,10005,20,Básico,670.2,Uruguay,Drama,2020-07-03,2
6,10006,37,Básico,346.6,Perú,Thriller,2019-07-26,1
7,10007,31,Estándar,974.6,Chile,Acción,2019-02-24,1
8,10008,36,Premium,1432.2,Colombia,Romance,2025-08-03,2
9,10009,37,Estándar,1375.4,Argentina,Thriller,2024-02-12,1


In [107]:
# Tipos de dato y valores no nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   object 
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   object 
 5   favorite_genre            7920 non-null   object 
 6   last_login_date           7840 non-null   object 
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 510.1+ KB


In [108]:
# Clasificación de variables
print("="*55)
print("VARIABLES CUANTITATIVAS (numéricas):"  )
print("  - user_id         : Discreta (identificador)")
print("  - age             : Discreta (años)")
print("  - monthly_watch_time_mins: Continua (minutos/mes)")
print("  - customer_support_tickets: Discreta (conteo)")
print()
print("VARIABLES CUALITATIVAS (categóricas):")
print("  - subscription_plan: Nominal (Básico/Estándar/Premium)")
print("  - country           : Nominal (7 países)")
print("  - favorite_genre    : Nominal (7 géneros)")
print("  - last_login_date   : Fecha (string → datetime)")

VARIABLES CUANTITATIVAS (numéricas):
  - user_id         : Discreta (identificador)
  - age             : Discreta (años)
  - monthly_watch_time_mins: Continua (minutos/mes)
  - customer_support_tickets: Discreta (conteo)

VARIABLES CUALITATIVAS (categóricas):
  - subscription_plan: Nominal (Básico/Estándar/Premium)
  - country           : Nominal (7 países)
  - favorite_genre    : Nominal (7 géneros)
  - last_login_date   : Fecha (string → datetime)


## 2. Estadísticos descriptivos

In [109]:
# Resumen numérico completo
df.describe().round(2)

,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.00,8160.00,7967.00,8160.00
mean,13995.43,34.10,1107.35,1.80
std,2310.81,14.51,5310.44,11.33
min,10000.00,-5.00,-120.00,-1.00
25%,11987.75,25.00,489.20,0.00
50%,13998.50,33.00,757.40,1.00
75%,15997.25,42.00,1045.70,1.00
max,17999.00,150.00,99999.00,150.00


**Observaciones clave de `describe()`:**
- `age`: min = -5, max = 150 
- `monthly_watch_time_mins`: min = -120 no puede ser valor negativo y max = 99 999 valor imposible por que el mes tiene 44640 minutos.
- `customer_support_tickets`: min = -1 imposible, max = 150 outlier extremo dado que el 75% tiene ≤ 1.

## 3. Datos faltantes

In [110]:
# Conteo y porcentaje de faltantes por columna
faltantes = df.isnull().sum()
porcentaje = (faltantes / len(df) * 100).round(2)
resumen = pd.DataFrame({'Faltantes': faltantes, '% del total': porcentaje})
print(resumen.sort_values('Faltantes', ascending=False))

print(f"\nTotal de celdas nulas: {faltantes.sum()} de {df.shape[0]*df.shape[1]} ({faltantes.sum()/(df.shape[0]*df.shape[1])*100:.2f}%)")

                          Faltantes  % del total
last_login_date                 320         3.92
favorite_genre                  240         2.94
monthly_watch_time_mins         193         2.37
user_id                           0         0.00
subscription_plan                 0         0.00
age                               0         0.00
country                           0         0.00
customer_support_tickets          0         0.00

Total de celdas nulas: 753 de 65280 (1.15%)


**Observaciones:**
- `last_login_date`: 3.92% de nulos. 
- `favorite_genre`: 2.94% de nulos.
- `monthly_watch_time_mins`: 2.37% nulos.
- `age`, `customer_support_tickets`, `subscription_plan`, `country`: 0 nulos explícitos.

## 4. Duplicados

In [111]:
# Registros completamente duplicados (excepto user_id)
n_dup = df.drop_duplicates(subset=df.columns.difference(['user_id'])).shape[0]
duplicados = len(df) - n_dup
print(f"Registros duplicados (ignorando user_id): {duplicados} ({duplicados/len(df)*100:.2f}%)")
print()
# Ver algunos duplicados
dup_mask = df.duplicated(subset=df.columns.difference(['user_id']), keep=False)
print("Ejemplo de filas duplicadas:")
print(df[dup_mask].sort_values('age').head(6).to_string())

Registros duplicados (ignorando user_id): 126 (1.54%)

Ejemplo de filas duplicadas:
      user_id  age subscription_plan  monthly_watch_time_mins   country favorite_genre last_login_date  customer_support_tickets
935     10935   13          Estándar                   1526.0  Colombia       Thriller      2024-06-21                         1
1420    11420   13            Básico                    202.2    Brasil        Comedia      2022-01-25                         1
1688    11688   13            Básico                    534.0      Perú          Crime      2025-11-22                         1
8028    10935   13          Estándar                   1526.0  Colombia       Thriller      2024-06-21                         1
8059    11420   13            Básico                    202.2    Brasil        Comedia      2022-01-25                         1
8107    11688   13            Básico                    534.0      Perú          Crime      2025-11-22                         1


## 5. Problemas en variables categóricas

In [112]:
# Inspección de variantes en cada categórica
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f"\n{'='*50}")
    print(f"  {col} — {df[col].nunique(dropna=False)} variantes únicas:")
    print(df[col].value_counts(dropna=False).to_string())


  subscription_plan — 15 variantes únicas:
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22

  country — 26 variantes únicas:
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
argentina       16
MEX             16
Chile           16
PER             16
Peru            15
BRA             15
chile           15
Mexico          15
brasil          13
perú            12
ARG             10
Argentina       10

  favorite_genre — 29 variantes únicas:
favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Romance        1

**Resumen de inconsistencias categóricas:**
- `subscription_plan`: 15 variantes para 3 valores reales Básico, Estándar y Premium. Problemas: mayúsculas, tildes, abreviaturas "Std"y "STANDARD", errores tipográficos "Premiun".
- `country`: 26 variantes para 7 países. Problemas: mayúsculas, idioma, abreviaturas ISO, espacios finales.
- `favorite_genre`: 28 variantes para 7 géneros, más 240 nulos. Problemas: mayúsculas, idioma, errores tipográficos, abreviaturas, equivalencias entre idiomas.

## 6. Problemas en variables numéricas

In [113]:
# Detección de valores imposibles
print("AGE:")
print(f"  Negativos (< 0): {(df['age'] < 0).sum()}")
print(f"  Mayores de 100: {(df['age'] > 100).sum()}")
print(f"  Rango observado: [{df['age'].min()}, {df['age'].max()}]")
invalid_age_count = ((df['age'] < 1) | (df['age'] > 100)).sum()
print(f"  Conteo de valores inválidos: {invalid_age_count}")

print("\nMONTHLY_WATCH_TIME_MINS:")
print(f"  Negativos (< 0): {(df['monthly_watch_time_mins'] < 0).sum()}")
print(f"  Nulos: {df['monthly_watch_time_mins'].isnull().sum()}")
print(f"  Máximo: {df['monthly_watch_time_mins'].max():.1f} (99 999 = valor centinela sospechoso)")

print("\nCUSTOMER_SUPPORT_TICKETS:")
print(f"  Negativos (< 0): {(df['customer_support_tickets'] < 0).sum()}")
print(f"  Distribución:")
print(df['customer_support_tickets'].value_counts().sort_index().to_string())
invalid_tickets_count = ((df['customer_support_tickets'] < 0) | (df['customer_support_tickets'] > 20)).sum()
print(f"  Conteo de valores inválidos: {invalid_tickets_count}")


AGE:
  Negativos (< 0): 21
  Mayores de 100: 53
  Rango observado: [-5, 150]
  Conteo de valores inválidos: 98

MONTHLY_WATCH_TIME_MINS:
  Negativos (< 0): 49
  Nulos: 193
  Máximo: 99999.0 (99 999 = valor centinela sospechoso)

CUSTOMER_SUPPORT_TICKETS:
  Negativos (< 0): 29
  Distribución:
customer_support_tickets
-1        29
 0      3628
 1      2885
 2      1179
 3       285
 4        73
 5        14
 99       35
 150      32
  Conteo de valores inválidos: 96


## 7. Problema en variable fecha

In [114]:
# Detectar fechas inválidas al parsear
fechas_parseadas = pd.to_datetime(df['last_login_date'], errors='coerce')
inválidas = df['last_login_date'].notna() & fechas_parseadas.isna()
print(f"Fechas inválidas (no parseables): {inválidas.sum()}")
print(f"Total no utilizables (nulos + inválidas): {fechas_parseadas.isna().sum()}")
print()
print("Ejemplos de fechas inválidas:")
print(df.loc[inválidas, 'last_login_date'].value_counts().head(10).to_string())

Fechas inválidas (no parseables): 449
Total no utilizables (nulos + inválidas): 769

Ejemplos de fechas inválidas:
last_login_date
2026-15-03       20
0000-00-00       17
not_available    14
31-02-2022       13
01-06-2025        2
2023/12/28        2
10-09-2022        2
20-03-2023        2
2023/05/31        2
2025/07/15        2


In [115]:
df['last_login_date'].isnull().sum()

np.int64(320)

## 8. Preguntas abiertas para el análisis

In [ ]:
print("""
PREGUNTAS QUE SURGEN DE LA INSPECCIÓN:
=======================================
1. ¿Cómo se distribuye el tiempo de visualización mensual de los usuarios y cuáles son sus principales características estadísticas?
2. ¿Cómo se distribuye la edad de los usuarios y qué características presenta esta variable?
3. ¿Cómo se distribuye el tiempo de visualización mensual según el plan de suscripción?
4. ¿Existe relación entre la edad y los minutos de visualización mensual?
5. ¿Existen diferencias en la cantidad promedio de tickets de soporte entre los distintos países y planes de suscripción?
""")


PREGUNTAS QUE SURGEN DE LA INSPECCIÓN:
1. ¿Cómo se distribuye el tiempo de visualización mensual según el plan de suscripción?
2. ¿Existe relación entre edad y minutos de visualización?
3. ¿Existen diferencias en la cantidad promedio de tickets de soporte entre los distintos países y planes de suscripción?
4. ¿Cómo se distribuye el tiempo de visualización mensual de los usuarios y qué características presenta esta variable?
5. ¿Los tickets de soporte están asociados a algún plan o país?



## Resumen de la inspección

| Problema detectado | Cantidad | Acción planeada |
|---|---|---|
| Duplicados exactos | 126 | Eliminar |
| Variantes en `subscription_plan` | 15 → 3 | Normalizar |
| Variantes en `country` | 26 → 7 | Normalizar |
| Variantes en `favorite_genre` | 28 → 7 | Normalizar |
| `age` imposibles | 96 | NaN → imputar (mediana) |
| `monthly_watch_time_mins` negativos | 49 | NaN → imputar |
| `monthly_watch_time_mins` nulos originales | 193 | Imputar por plan |
| `monthly_watch_time_mins` extremo (99 999) | múltiples | Winsorizar k=3 |
| `customer_support_tickets` negativos | 29 | NaN → imputar |
| `customer_support_tickets` extremos | 96 | Winsorizar k=3 |
| `last_login_date` nulos | 320 | Dejar NaT |
| `last_login_date` inválidas | 449 | Dejar NaT |
| `favorite_genre` nulos | 240 | Imputar con 'otros' |

